In [3]:
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [5]:
print("Step 1: Loading Dataset & Tokenizer...")
dataset = load_dataset("SetFit/ag_news")
train_subset = dataset["train"].shuffle(seed=42).select(range(1000)) # Clean subset for fast execution
test_subset = dataset["test"].shuffle(seed=42).select(range(200))

Step 1: Loading Dataset & Tokenizer...


train.jsonl:   0%|          | 0.00/33.8M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
def tokenize_fn(ex):
    return tokenizer(ex["text"], padding="max_length", truncation=True, max_length=128)

tok_train = train_subset.map(tokenize_fn, batched=True)
tok_test = test_subset.map(tokenize_fn, batched=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [7]:
print("Step 2: Training Model...")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)
acc_metric = evaluate.load("accuracy")

Step 2: Training Model...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return acc_metric.compute(predictions=predictions, references=labels)

args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
)

trainer = Trainer(model=model, args=args, train_dataset=tok_train, eval_dataset=tok_test, compute_metrics=compute_metrics)
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.771128,0.815000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=63, training_loss=1.040620834108383, metrics={'train_runtime': 1363.9329, 'train_samples_per_second': 0.733, 'train_steps_per_second': 0.046, 'total_flos': 65778945024000.0, 'train_loss': 1.040620834108383, 'epoch': 1.0})

In [10]:
print("Step 3: Saving Model locally...")
# Save everything so our web app can load it instantly later
model.save_pretrained("./my_saved_bert_model")
tokenizer.save_pretrained("./my_saved_bert_model")
print("All Done! Model is saved successfully.")

Step 3: Saving Model locally...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

All Done! Model is saved successfully.


In [12]:
%%writefile app.py
import streamlit as st
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

st.set_page_config(page_title="BERT News Classifier", layout="centered")
st.title("📰 News Topic Classifier Using Fine-Tuned BERT")
st.write("This application uses your fine-tuned BERT model to predict news categories.")

# Load the saved model instantly from your local folder
@st.cache_resource
def load_saved_model():
    tokenizer = AutoTokenizer.from_pretrained("./my_saved_bert_model")
    model = AutoModelForSequenceClassification.from_pretrained("./my_saved_bert_model")
    return tokenizer, model

tokenizer, model = load_saved_model()

st.subheader("Test the Model with Live News Headlines")
user_input = st.text_area("Enter a news headline or short summary here:")
label_names = ["World", "Sports", "Business", "Sci/Tech"]

if st.button("Predict Category"):
    if user_input.strip() != "":
        # Process input text
        inputs = tokenizer(user_input, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # Make prediction
        outputs = model(**inputs)
        prediction = np.argmax(outputs.logits.detach().cpu().numpy(), axis=-1)[0]
        result = label_names[prediction]

        # Display results cleanly
        st.metric(label="Predicted Topic Category", value=result)
    else:
        st.warning("Please enter some text before clicking predict.")

Overwriting app.py


In [17]:
!streamlit run app.py



2026-06-08 05:39:18.210 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.59.154.43:8501

  Stopping...
  Stopping...


In [18]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 5s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

In [19]:
import urllib
print("Your Tunnel Password / IP is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

Your Tunnel Password / IP is: 34.59.154.43


In [20]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇

⠏⠋⠙⠹⠸⠼your url is: https://tough-days-reply.loca.lt
2026-06-08 05:43:45.404 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.59.154.43:8501

  Stopping...
^C
